In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف تمام توابع تحلیل (10 تابع - 3 مجموعه)
# ============================================================================

# =====================================================================
# مجموعه ۱: خوشه‌بندی بیرینگ (Bearing) - 4 الگوریتم
# =====================================================================

def analysis_bearing_isolation_forest(file_path, output_filename):
    """تحلیل با Isolation Forest برای بیرینگ"""
    print(f"\n{'='*60}")
    print(f"🔄 [بیرینگ-1] Isolation Forest")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    contamination = 0.1
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        model = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
        df_model['Behavior_Cluster'] = model.fit_predict(scaled_data)
        
        anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
        print(f"   تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
        
        scores = model.decision_function(scaled_data)
        min_s, max_s = scores.min(), scores.max()
        if max_s - min_s == 0:
            df_model['Degradation_Index'] = 0
        else:
            df_model['Degradation_Index'] = 1.0 - (scores - min_s) / (max_s - min_s)
        
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        expert_data = df_model[df_model['Behavior_Cluster'] == -1].copy()
        expert_data = expert_data.sort_values(by='Degradation_Index', ascending=False)
        expert_data['Expert_Label'] = ""
        expert_data['Expert_Comments'] = ""
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        expert_data.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_bearing_dbscan(file_path, output_filename):
    """تحلیل با DBSCAN برای بیرینگ"""
    print(f"\n{'='*60}")
    print(f"🔄 [بیرینگ-2] DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    eps, min_samples = 0.5, 5
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        model = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        df_model['Behavior_Cluster'] = cluster_labels
        
        n_clusters = len([x for x in set(cluster_labels) if x != -1])
        n_noise = sum(1 for x in cluster_labels if x == -1)
        print(f"   تعداد خوشه‌ها: {n_clusters}, نقاط نویز: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")
        
        # محاسبه شاخص تخریب (ساده شده)
        core_mask = np.zeros(len(scaled_data), dtype=bool)
        core_mask[model.core_sample_indices_] = True
        
        unique_clusters = set(cluster_labels) - {-1}
        cluster_centers = {}
        for cluster_id in unique_clusters:
            cluster_core_points = scaled_data[(cluster_labels == cluster_id) & core_mask]
            if len(cluster_core_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_core_points, axis=0)
            else:
                cluster_all_points = scaled_data[cluster_labels == cluster_id]
                if len(cluster_all_points) > 0:
                    cluster_centers[cluster_id] = np.mean(cluster_all_points, axis=0)
                else:
                    cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        max_distance = 0
        degradation = np.zeros(len(scaled_data))
        for i in range(len(scaled_data)):
            cluster_id = cluster_labels[i]
            if cluster_id == -1:
                degradation[i] = 0
            else:
                center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                degradation[i] = np.linalg.norm(scaled_data[i] - center)
                if degradation[i] > max_distance:
                    max_distance = degradation[i]
        
        for i in range(len(scaled_data)):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        df_model['Degradation_Index'] = degradation
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        def get_health_status(row):
            if row['Behavior_Cluster'] == -1 or row['Degradation_Index'] > 0.95:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 0.85:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_dt = df_model['date'].max()
            start_date = last_dt - timedelta(days=30)
            final_df = df_model[df_model['date'] >= start_date].copy()
        else:
            final_df = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_df.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_bearing_lof(file_path, output_filename):
    """تحلیل با LOF برای بیرینگ"""
    print(f"\n{'='*60}")
    print(f"🔄 [بیرینگ-3] LOF")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    n_neighbors, contamination = 20, 0.05
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)
        df_model['Degradation_Index'] = -lof.negative_outlier_factor_
        
        anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
        print(f"   تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.3:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            start_date = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= start_date].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_bearing_kmeans(file_path, output_filename):
    """تحلیل با K-Means برای بیرینگ"""
    print(f"\n{'='*60}")
    print(f"🔄 [بیرینگ-4] K-Means")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    n_clusters = 3
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)
        
        cluster_counts = df_model['Behavior_Cluster'].value_counts().sort_index()
        print(f"   تعداد خوشه‌ها: {len(cluster_counts)}")
        for cluster_id, count in cluster_counts.items():
            print(f"   خوشه {cluster_id}: {count:,} رکورد ({count/len(df_model)*100:.2f}%)")
        
        distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
        df_model['Degradation_Index'] = distances
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.5:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.5:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            one_month_ago = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= one_month_ago].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# =====================================================================
# مجموعه ۲: خوشه‌بندی ژنراتور (Generator) - 3 الگوریتم
# =====================================================================

def analysis_generator_dbscan(file_path, output_filename):
    """تحلیل با DBSCAN برای ژنراتور"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-1] DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    eps, min_samples = 0.5, 5
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        model = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        df_model['Behavior_Cluster'] = cluster_labels
        
        n_clusters = len([x for x in set(cluster_labels) if x != -1])
        n_noise = sum(1 for x in cluster_labels if x == -1)
        print(f"   تعداد خوشه‌ها: {n_clusters}, نقاط نویز: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")
        
        # شاخص تخریب ساده شده
        unique_clusters = set(cluster_labels) - {-1}
        cluster_centers = {}
        for cluster_id in unique_clusters:
            cluster_points = scaled_data[cluster_labels == cluster_id]
            if len(cluster_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_points, axis=0)
            else:
                cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        degradation = np.zeros(len(scaled_data))
        max_distance = 0
        for i in range(len(scaled_data)):
            cluster_id = cluster_labels[i]
            if cluster_id == -1:
                degradation[i] = 0
            else:
                center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                degradation[i] = np.linalg.norm(scaled_data[i] - center)
                if degradation[i] > max_distance:
                    max_distance = degradation[i]
        
        for i in range(len(scaled_data)):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        df_model['Degradation_Index'] = degradation
        
        def get_health_status(row):
            if row['Behavior_Cluster'] == -1 or row['Degradation_Index'] > 0.95:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 0.85:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_dt = df_model['date'].max()
            start_date = last_dt - timedelta(days=30)
            final_df = df_model[df_model['date'] >= start_date].copy()
        else:
            final_df = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_df.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_generator_lof(file_path, output_filename):
    """تحلیل با LOF برای ژنراتور"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-2] LOF")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    n_neighbors, contamination = 20, 0.05
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)
        df_model['Degradation_Index'] = -lof.negative_outlier_factor_
        
        anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
        print(f"   تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.3:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            start_date = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= start_date].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_generator_kmeans(file_path, output_filename):
    """تحلیل با K-Means برای ژنراتور"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-3] K-Means")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    n_clusters = 3
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)
        
        distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
        df_model['Degradation_Index'] = distances
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.5:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.5:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            one_month_ago = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= one_month_ago].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# =====================================================================
# مجموعه ۳: خوشه‌بندی روغن‌کاری (Lubrication) - 3 الگوریتم
# =====================================================================

def analysis_lubrication_dbscan(file_path, output_filename):
    """تحلیل با DBSCAN برای روغن‌کاری"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-1] DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    eps, min_samples = 0.5, 5
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        model = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        df_model['Behavior_Cluster'] = cluster_labels
        
        n_clusters = len([x for x in set(cluster_labels) if x != -1])
        n_noise = sum(1 for x in cluster_labels if x == -1)
        print(f"   تعداد خوشه‌ها: {n_clusters}, نقاط نویز: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")
        
        unique_clusters = set(cluster_labels) - {-1}
        cluster_centers = {}
        for cluster_id in unique_clusters:
            cluster_points = scaled_data[cluster_labels == cluster_id]
            if len(cluster_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_points, axis=0)
            else:
                cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        degradation = np.zeros(len(scaled_data))
        max_distance = 0
        for i in range(len(scaled_data)):
            cluster_id = cluster_labels[i]
            if cluster_id == -1:
                degradation[i] = 0
            else:
                center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                degradation[i] = np.linalg.norm(scaled_data[i] - center)
                if degradation[i] > max_distance:
                    max_distance = degradation[i]
        
        for i in range(len(scaled_data)):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        df_model['Degradation_Index'] = degradation
        
        def get_health_status(row):
            if row['Behavior_Cluster'] == -1 or row['Degradation_Index'] > 0.95:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 0.85:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_dt = df_model['date'].max()
            start_date = last_dt - timedelta(days=30)
            final_df = df_model[df_model['date'] >= start_date].copy()
        else:
            final_df = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_df.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_lubrication_lof(file_path, output_filename):
    """تحلیل با LOF برای روغن‌کاری"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-2] LOF")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    n_neighbors, contamination = 20, 0.05
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)
        df_model['Degradation_Index'] = -lof.negative_outlier_factor_
        
        anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
        print(f"   تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.3:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            start_date = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= start_date].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_lubrication_kmeans(file_path, output_filename):
    """تحلیل با K-Means برای روغن‌کاری"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-3] K-Means")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    n_clusters = 3
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)
        
        distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
        df_model['Degradation_Index'] = distances
        
        def get_health_status(row):
            if row['Degradation_Index'] > 2.5:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.5:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            one_month_ago = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= one_month_ago].copy()
        else:
            final_output = df_model
        
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 10 وظیفه
# ============================================================================

def get_all_jobs():
    """تعریف تمام ۱۰ وظیفه تحلیل - 3 مجموعه"""
    jobs = []
    
    # مجموعه ۱: بیرینگ (4 الگوریتم)
    bearing_base = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    bearing_out_base = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output'
    
    jobs.extend([
        {'name': 'Bearing-1: Isolation Forest', 'function': analysis_bearing_isolation_forest,
         'file_path': bearing_base, 'output_filename': f'{bearing_out_base}1.xlsx'},
        {'name': 'Bearing-2: DBSCAN', 'function': analysis_bearing_dbscan,
         'file_path': bearing_base, 'output_filename': f'{bearing_out_base}2.xlsx'},
        {'name': 'Bearing-3: LOF', 'function': analysis_bearing_lof,
         'file_path': bearing_base, 'output_filename': f'{bearing_out_base}3.xlsx'},
        {'name': 'Bearing-4: K-Means', 'function': analysis_bearing_kmeans,
         'file_path': bearing_base, 'output_filename': f'{bearing_out_base}4.xlsx'}
    ])
    
    # مجموعه ۲: ژنراتور (3 الگوریتم)
    gen_base = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    gen_out_base = r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output'
    
    jobs.extend([
        {'name': 'Generator-1: DBSCAN', 'function': analysis_generator_dbscan,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}2.xlsx'},
        {'name': 'Generator-2: LOF', 'function': analysis_generator_lof,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}3.xlsx'},
        {'name': 'Generator-3: K-Means', 'function': analysis_generator_kmeans,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}4.xlsx'}
    ])
    
    # مجموعه ۳: روغن‌کاری (3 الگوریتم)
    lub_base = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
    lub_out_base = r'outputs\G11\dsas_g11_lubrication_system_clustering\clustering\dsas_g11_lubrication\clustering_g11_lubrication_output'
    
    jobs.extend([
        {'name': 'Lubrication-1: DBSCAN', 'function': analysis_lubrication_dbscan,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}2.xlsx'},
        {'name': 'Lubrication-2: LOF', 'function': analysis_lubrication_lof,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}3.xlsx'},
        {'name': 'Lubrication-3: K-Means', 'function': analysis_lubrication_kmeans,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}4.xlsx'}
    ])
    
    return jobs


def run_all_analyses():
    """اجرای تمام ۱۰ تحلیل به ترتیب"""
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌های خوشه‌بندی (۱۰ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 مجموعه‌ها:")
    print("   1. بیرینگ (Bearing) - ۴ الگوریتم")
    print("   2. ژنراتور (Generator) - ۳ الگوریتم")
    print("   3. روغن‌کاری (Lubrication) - ۳ الگوریتم")
    print("="*80)
    print("📊 مجموع: ۱۰ خروجی")
    print("="*80)
    
    jobs = get_all_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    print("\n📋 جزئیات:")
    for i, r in enumerate(results, 1):
        status = "✅" if r['success'] else "❌"
        print(f"   {i:2d}. {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)"""
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد")
    print("📋 شامل ۳ مجموعه × (۳ یا ۴ الگوریتم) = ۱۰ خروجی")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            if current_time in ["13:27", "13:47"]:
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    results = run_all_analyses()
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    time.sleep(60)
            
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع تحلیل خوشه‌بندی (۳ مجموعه یکپارچه)")
        print("="*80)
        print("📋 مجموعه‌ها و خروجی‌ها:")
        print("   ┌─────────────────────────────────────────────────────────┐")
        print("   │  مجموعه ۱: بیرینگ (Bearing)          → ۴ خروجی      │")
        print("   │  مجموعه ۲: ژنراتور (Generator)       → ۳ خروجی      │")
        print("   │  مجموعه ۳: روغن‌کاری (Lubrication)   → ۳ خروجی      │")
        print("   └─────────────────────────────────────────────────────────┘")
        print("   مجموع: ۱۰ فایل خروجی")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع تحلیل خوشه‌بندی (۳ مجموعه یکپارچه)
📋 مجموعه‌ها و خروجی‌ها:
   ┌─────────────────────────────────────────────────────────┐
   │  مجموعه ۱: بیرینگ (Bearing)          → ۴ خروجی      │
   │  مجموعه ۲: ژنراتور (Generator)       → ۳ خروجی      │
   │  مجموعه ۳: روغن‌کاری (Lubrication)   → ۳ خروجی      │
   └─────────────────────────────────────────────────────────┘
   مجموع: ۱۰ فایل خروجی
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد
📋 شامل ۳ مجموعه × (۳ یا ۴ الگوریتم) = ۱۰ خروجی
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-09 13:27:20

🚀 شروع اجرای همه تحلیل‌های خوشه‌بندی (۱۰ وظیفه)
📅 زمان: 2026-07-09 13:27:20
📋 مجموعه‌ها:
   1. بیرینگ (Bearing) - ۴ الگوریتم
   2. ژنراتور (Generator) - ۳ الگوریتم
   3. روغن‌کاری (Lubrication) - ۳ الگوریتم
📊 مجموع: ۱۰ خروجی

################################################################################
# وظیفه 1 از 10: Beari